# Replay

In [ ]:
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver
from typing_extensions import TypedDict, NotRequired
from langchain_core.utils.uuid import uuid7
from rich import print

class State(TypedDict):
    topic: NotRequired[str]
    joke: NotRequired[str]


def generate_topic(state: State):
    return {"topic": "socks in the dryer"}


def write_joke(state: State):
    return {"joke": f"Why do {state['topic']} disappear? They elope!"}


checkpointer = InMemorySaver()
graph = (
    StateGraph(State)
    .add_node("generate_topic", generate_topic)
    .add_node("write_joke", write_joke)
    .add_edge(START, "generate_topic")
    .add_edge("generate_topic", "write_joke")
    .compile(checkpointer=checkpointer)
)

# Step 1: Run the graph
config = {"configurable": {"thread_id": str(uuid7())}}
result = graph.invoke({}, config)

# Step 2: Find a checkpoint to replay from
history = list(graph.get_state_history(config))
# History is in reverse chronological order
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")

# Step 3: Replay from a specific checkpoint
# Find the checkpoint before write_joke
before_joke = next(s for s in history if s.next == ("write_joke",))
replay_result = graph.invoke(None, before_joke.config)
# write_joke re-executes (runs again), generate_topic does not

print("-"*50)
# Step 2: Find a checkpoint to replay from
history = list(graph.get_state_history(config))
# History is in reverse chronological order
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")

next=(), checkpoint_id=1f15f726-ad2c-65c9-8002-19907675392a

next=('write_joke',), checkpoint_id=1f15f726-ad28-6ab3-8001-beeb1d8c8858

next=('generate_topic',), checkpoint_id=1f15f726-ad26-638e-8000-afb3318b9d05

next=('__start__',), checkpoint_id=1f15f726-ad23-6c8c-bfff-e99a692d178d

--------------------------------------------------

...

next=('write_joke',), checkpoint_id=1f15f726-ad52-669a-8002-1adef98d45e0

next=(), checkpoint_id=1f15f726-ad2c-65c9-8002-19907675392a

next=('write_joke',), checkpoint_id=1f15f726-ad28-6ab3-8001-beeb1d8c8858

next=('generate_topic',), checkpoint_id=1f15f726-ad26-638e-8000-afb3318b9d05

next=('__start__',), checkpoint_id=1f15f726-ad23-6c8c-bfff-e99a692d178d

In [14]:
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver
from typing_extensions import TypedDict, NotRequired
from langchain_core.utils.uuid import uuid7
from rich import print

class State(TypedDict):
    topic: NotRequired[str]
    joke: NotRequired[str]


def generate_topic(state: State):
    return {"topic": "socks in the dryer"}


def write_joke(state: State):
    return {"joke": f"Why do {state['topic']} disappear? They elope!"}


checkpointer = InMemorySaver()
graph = (
    StateGraph(State)
    .add_node("generate_topic", generate_topic)
    .add_node("write_joke", write_joke)
    .add_edge(START, "generate_topic")
    .add_edge("generate_topic", "write_joke")
    .compile(checkpointer=checkpointer)
)

# Step 1: Run the graph
config = {"configurable": {"thread_id": str(uuid7())}}
result = graph.invoke({}, config)

# Find checkpoint before write_joke
history = list(graph.get_state_history(config))
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")
    
    
before_joke = next(s for s in history if s.next == ("write_joke",))
print(f"previous state :: {before_joke}")

print(f"Checkpoint before write_joke: {before_joke.config['configurable']['checkpoint_id']}")
# Fork: update state to change the topic
fork_config = graph.update_state(
    before_joke.config,
    values={"topic": "chickens"},
)

# Resume from the fork — write_joke re-executes with the new topic
fork_result = graph.invoke(None, fork_config)
print(fork_result["joke"])  # A joke about chickens, not socks

print(f"new state :: {fork_result}")
# Find checkpoint before write_joke
history = list(graph.get_state_history(config))
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")
    
 

next=(), checkpoint_id=1f15f72d-9b4f-6f61-8002-08e936e4d904

next=('write_joke',), checkpoint_id=1f15f72d-9b4d-680a-8001-6a89e681411b

next=('generate_topic',), checkpoint_id=1f15f72d-9b4b-608f-8000-b506776a6060

next=('__start__',), checkpoint_id=1f15f72d-9b47-65b4-bfff-be43e37f76b5

previous state :: StateSnapshot(values={'topic': 'socks in the dryer'}, next=('write_joke',), 
config={'configurable': {'thread_id': '019e8e8f-84c7-7b02-a93b-f6983cc82810', 'checkpoint_ns': '', 'checkpoint_id':
'1f15f72d-9b4d-680a-8001-6a89e681411b'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, 
created_at='2026-06-03T17:37:10.861415+00:00', parent_config={'configurable': {'thread_id': 
'019e8e8f-84c7-7b02-a93b-f6983cc82810', 'checkpoint_ns': '', 'checkpoint_id': 
'1f15f72d-9b4b-608f-8000-b506776a6060'}}, tasks=(PregelTask(id='494c1002-f161-ffd3-1854-1b3f63d337e9', 
name='write_joke', path=('__pregel_pull', 'write_joke'), error=None, interrupts=(), state=None, result={'joke': 
'Why do socks in the dryer disappear? They elope!'}),), interrupts=())

...

next=('write_joke',), checkpoint_id=1f15f72d-9b7d-6483-8002-6d47746f0e08

next=(), checkpoint_id=1f15f72d-9b4f-6f61-8002-08e936e4d904

next=('write_joke',), checkpoint_id=1f15f72d-9b4d-680a-8001-6a89e681411b

next=('generate_topic',), checkpoint_id=1f15f72d-9b4b-608f-8000-b506776a6060

next=('__start__',), checkpoint_id=1f15f72d-9b47-65b4-bfff-be43e37f76b5